In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

C:\Users\DELL\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\DELL\anaconda3\lib\site-packages\numpy\.libs\libopenblas.WCDJNK7YVMPZQ2ME2ZZHJJRJ3JIKNDB7.gfortran-win_amd64.dll
C:\Users\DELL\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


ImportError: Unable to import required dependencies:
numpy: cannot import name 'set_array_function_like_doc' from 'numpy.core.overrides' (C:\Users\DELL\anaconda3\lib\site-packages\numpy\core\overrides.py)

In [ ]:
dataset = pd.read_csv("archive/city_day.csv")
dataset

In [ ]:
dataset = dataset.drop(columns=['City','Date'])
dataset

In [ ]:
dataset.isna().sum()/dataset.shape[0]

In [ ]:
dataset = dataset[dataset['AQI'].isna()==False]
dataset

In [ ]:
dataset.isna().sum()/dataset.shape[0]

In [ ]:
columns_to_remove_null_rows = [i for i in dataset.columns if dataset[i].isna().sum()/dataset.shape[0]<=0.1 and dataset[i].isna().sum()>0]

columns_to_impute = [i for i in dataset.columns if i not in columns_to_remove_null_rows and dataset[i].isna().sum()>0]

print("Columns in which rows will be removed:", columns_to_remove_null_rows)
print("Columns which will be imputed:", columns_to_impute)

In [ ]:
from sklearn.impute import SimpleImputer

def remove_rows(data, column):
  return data[data[column].isna()==False]

for i in columns_to_remove_null_rows:
  dataset = remove_rows(dataset, i)

imputer = SimpleImputer()
dataset[columns_to_impute] = imputer.fit_transform(dataset[columns_to_impute])

dataset

In [ ]:
dataset.isna().sum()/dataset.shape[0]

In [ ]:
dataset[[i for i in dataset.columns if i not in ["AQI_Bucket","AQI"]]].hist(bins=30, figsize=(15, 10))
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

dataset_norm_test = dataset[[i for i in dataset.columns if i not in ["AQI_Bucket","AQI"]]].copy()
scaler = StandardScaler()
dataset_norm_test = pd.DataFrame(scaler.fit_transform(dataset_norm_test),columns=dataset_norm_test.columns)
dataset_norm_test.hist(bins=30, figsize=(15, 10))
plt.show()

In [ ]:
import scipy.stats as stats

dataset_yeo_test = dataset[[i for i in dataset.columns if i not in ["AQI_Bucket","AQI"]]].copy()

for i in dataset_yeo_test.columns:
  yeo_t,param = stats.yeojohnson(dataset_yeo_test[i])
  plt.hist(yeo_t,bins=25)
  plt.xlabel(i)
  plt.show()

In [ ]:
columns_to_binarize = ['Benzene','Toluene','Xylene']
columns_to_yeo = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3']

print('Columns to binarize:',columns_to_binarize)
print('Columns to yeo transform: ', columns_to_yeo)

In [ ]:
dataset[columns_to_binarize].hist(bins=60, figsize=(15, 10))
plt.show()

In [ ]:
binarize_threshold = {
    'Benzene' : 30,
    'Xylene' : 30,
    'Toluene' : 30,
}

def binarize(value, thresold):
  if value<=thresold:
    return 0
  return 1

for col in columns_to_binarize:
  new_col = col+"_binarized"
  dataset[new_col] = dataset[col].apply(lambda x: binarize(x, binarize_threshold[col]))

dataset

In [ ]:
yeo_transform_params = {}

for col in columns_to_yeo:
  yeo_t,param = stats.yeojohnson(dataset_yeo_test[col])
  yeo_transform_params[col] = param
  dataset[col] = yeo_t

dataset

In [ ]:
yeo_transform_params

In [ ]:
import seaborn as sns

dataset_corr = dataset[[i for i in dataset.columns if i!="AQI_Bucket"]]

plt.figure(figsize=(12,10))
cor = dataset_corr.corr()
sns.heatmap(cor, annot=True, cmap=plt.cm.Reds)
plt.show()

In [ ]:
dataset.drop(columns = columns_to_binarize+['NOx'],inplace=True)

In [ ]:
dataset['AQI'] = np.log(dataset['AQI'])
dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dataset[[i for i in dataset.columns if i not in ["AQI","AQI_Bucket"]]], 
    dataset["AQI"], 
    test_size=0.2, 
    random_state=100
    )

X_train

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train),columns=X_train.columns)

X_test = pd.DataFrame(scaler.transform(X_test),columns=X_test.columns)

X_train

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

# Initialising the RNN
regressor = Sequential()

# Adding the input layerand the LSTM layer
regressor.add(LSTM(units = 8, activation = 'relu', input_shape = (None, 1)))


# Adding the output layer
regressor.add(Dense(units = 1))

# Compiling the RNN
regressor.compile(optimizer = 'adam', loss = 'mean_squared_error')

# Fitting the RNN to the Training set
regressor.fit(X_train, y_train, batch_size = 10, epochs = 10, verbose = 0)

In [ ]:
inputs = X_test
y_pred = regressor.predict(inputs)
#y_pred = ss.inverse_transform(y_pred)

In [ ]:
plt.figure
plt.plot(y_test, color = 'red', label = 'Real Values')
plt.plot(y_pred, color = 'blue', label = 'Predicted Values')
plt.title('Prediction Values')
plt.xlabel('Test Value')
plt.ylabel('Predicted Value')
plt.legend()
plt.show()

In [ ]:
print(y_test)

In [ ]:
print(y_pred)

In [ ]:
print(X_test)

In [ ]:
inputs=X_test.iloc[0]
y_pred = regressor.predict(inputs)
print(y_pred)